# The geography gap
**The Rental Price Puzzle · Generalization stress test**  
Cyril James Lunzaga

A separate deterministic experiment fits the frozen Random Forest specification on 43 state codes and evaluates on eight withheld state codes. The geographic train/test split is not the same test population as the primary split.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "portfolio" / "tables").is_dir():
    project_root = project_root.parent
tables = project_root / "portfolio" / "tables"
assert tables.is_dir(), "Open from the repository root or notebooks folder, and generate portfolio tables first"
metrics = pd.read_csv(tables / "heldout_model_comparison.csv")
state_errors = pd.read_csv(tables / "unseen_state_errors.csv")
tails = pd.read_csv(tables / "training_defined_tail_sensitivity.csv")
geo = metrics.loc[metrics["evaluation"].eq("geographic")].set_index("model")
forest = geo.loc["random_forest_log_rent"]
baseline = geo.loc["median_baseline"]
print(f"Forest geographic MAE: ${forest['mae_usd']:,.2f}")
print(f"Baseline geographic MAE: ${baseline['mae_usd']:,.2f}")
print(f"Relative extra MAE: {(forest['mae_usd'] / baseline['mae_usd'] - 1) * 100:.1f}%")
print(f"Forest geographic R²: {forest['r2']:.3f}")
assert forest["mae_usd"] > baseline["mae_usd"]

## Errors across withheld state codes

The chart reports aggregate error for state groups with sufficient held-out sample size. Differences can arise from local rent structures, unseen categorical labels, and uneven advertising-source coverage. They are not unbiased estimates of each state's real rental market.

In [ ]:
visible = state_errors.head(12).sort_values("mae_usd")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(visible["state"].astype(str), visible["mae_usd"])
ax.set(xlabel="Mean absolute error (USD/month)", title="Withheld-state subgroup error")
fig.tight_layout()
plt.show()
display(state_errors)

## Why different error metrics matter

The Random Forest's geographic RMSE and R² are slightly better than its baseline, but its MAE is worse and both R² scores are negative. The study chose models using MAE, so the geographic MAE disadvantage cannot be described as successful generalization. Retuning on these withheld states would compromise their independence.

In [ ]:
display(metrics.loc[metrics["evaluation"].eq("geographic"), ["model", "heldout_listings", "mae_usd", "rmse_usd", "r2"]])
display(tails.loc[tails["evaluation"].eq("geographic")])

**Conclusion.** The data support useful prediction for group-separated advertisements from represented state markets. They do not support reliable transfer to newly encountered states. Future geography-aware model research would require a new, independently reserved evaluation sample.